
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Task 1- Data Quality Assessment

In this task, we will assess the quality of the loan dataset by checking for missing values, duplicates, and potential outliers. This step is critical to ensure that our data is clean and ready for further analysis in subsequent tasks.

**Objectives:**

- Identify and report missing values in each column.
- Detect duplicate rows.
- Perform basic outlier detection on numerical columns.

## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need to use one of the following Databricks runtime(s): **17.3.x-cpu-ml-scala2.13**

## Classroom Setup
Before starting the demo, run the provided classroom setup script. This script will define configuration variables necessary for the demo. Execute the following cell:

In [0]:
%run ../../Includes/Classroom-Setup-1.1demo

**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")


## Task Outline
In this task, we will:

- Load the loan dataset.
- Check for missing values in each column.
- Identify duplicate rows in the dataset.
- Detect potential outliers in numerical columns.

###Step 1: Load Loan Data
In this step, we will load the loan dataset from a Delta table. We will then inspect the data to understand its structure and ensure it has been loaded correctly.

**Instructions:**

- Define the path to the Delta table containing loan data.
- Use the Spark DataFrame API to load the data and display a sample for verification.

In [0]:
# Define the dataset path
dataset_path = f"{DA.paths.datasets.banking}/banking/loan-clean.csv"

# Load the loan dataset
loan_data = spark.read.format('csv').option('header', 'true').load(dataset_path)

# Display the first few rows to inspect the data
display(loan_data)


###Step 2: Data Quality Checks
We will perform three main data quality checks: identifying missing values, detecting duplicates, and finding potential outliers.


####Step 2.1: Check for Missing Values
**Instructions:**

- Use Spark functions to count missing values in each column.
- Display columns with missing values for further analysis.

In [0]:
from pyspark.sql.functions import count, when, col

# Check for missing values in each column
missing_values = loan_data.select([count(when(col(c).isNull(), c)).alias(c) for c in loan_data.columns])

# Display columns with missing values
display(missing_values)

####Step 2.2: Detect Duplicate Rows
**Instructions:**
- Use the `groupBy` and `count` functions to detect duplicate rows.
- Count the number of duplicate rows in the dataset.

In [0]:
# Check for duplicate rows
duplicates = loan_data.groupBy(loan_data.columns).count().filter("count > 1").count()
print(f"Number of duplicate rows: {duplicates}")

####Step 2.3: Outlier Detection in Numerical Columns
**Instructions:**

- Calculate the mean and standard deviation for numerical columns.
- Identify outliers as values that are more than three standard deviations from the mean.

In [0]:
from pyspark.sql.functions import mean, stddev

# List to store columns with detected outliers
outlier_columns = []

# Detect outliers in numerical columns
for column, dtype in loan_data.dtypes:
    if dtype in ['double', 'int']:
        mean_val = loan_data.select(mean(col(column))).collect()[0][0]
        stddev_val = loan_data.select(stddev(col(column))).collect()[0][0]
        outliers = loan_data.filter((col(column) > mean_val + 3 * stddev_val) | (col(column) < mean_val - 3 * stddev_val))
        if outliers.count() > 0:
            outlier_columns.append(column)

# Display columns with outliers
print("Columns with outliers:", outlier_columns)


###Step 3: Log Data Quality Report
We will save a data quality report with the results from the above checks, including missing values, duplicate rows, and columns with outliers.

In [0]:
# Save data quality report
with open("./data_quality_report.txt", "w") as f:
    f.write("Data Quality Report\n")
    f.write("===================\n")
    f.write("Missing Values:\n")
    missing_values_list = missing_values.collect()
    for row in missing_values_list:
        f.write(f"{row}\n")
    f.write(f"\nDuplicate rows: {duplicates}\n")
    f.write(f"Columns with outliers: {outlier_columns}\n")

print("Data Quality Report saved to ./data_quality_report.txt")


###Step 4: Set Conditional Flag for Data Quality Issues
Finally, we will set a flag to indicate if any data quality issues were found, which will guide the next steps in the pipeline.

In [0]:
data_quality_issues = bool(
    sum(missing_values.selectExpr(f"sum(`{col}`)").collect()[0][0] for col in missing_values.columns) > 0 or 
    duplicates > 0 or 
    outlier_columns
)
print(f"Data Quality Issues Found: {data_quality_issues}")


##Conclusion
In this notebook, we:

- Loaded and inspected the loan dataset.
- Checked for missing values, duplicates, and potential outliers.
- Generated a data quality report and saved it to the Databricks file system.
- Set a conditional flag to indicate the presence of data quality issues.

This data quality assessment ensures that the dataset is prepared for further analysis. If data quality issues are found, they will trigger an alternative path in the workflow pipeline for additional review or corrective actions.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>